# 03 Lead History Merge

This notebook merges the reviewed history file with the reviewed new assigned file.

It checks:
- exact duplicate rows
- conflicting values under the same final `pwsid + monitoring_end_date`

Only when both checks pass does it export `All_history_data_clean.csv`.


In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('.').resolve()
MERGE_OUTPUT_DIR = BASE_DIR / 'merge_pipeline'
MERGE_OUTPUT_DIR.mkdir(exist_ok=True)

history_file_name = input(
    'Enter the history file name in the current folder '
    '(example: lead_90th_history_no_duplicate_renamed.csv): '
).strip()

if not history_file_name:
    raise ValueError('You must enter a history file name.')

HISTORY_FILE_PATH = BASE_DIR / history_file_name

NEW_ASSIGNED_FILE_PATH = (
    BASE_DIR / 'assign_pipeline' / 'Public_Water_Supply_90th_Percentiles_2025_with_assigned_pwsid.csv'
)

if not HISTORY_FILE_PATH.exists():
    raise FileNotFoundError(f'History file not found: {HISTORY_FILE_PATH}')

if not NEW_ASSIGNED_FILE_PATH.exists():
    raise FileNotFoundError(f'New assigned file not found: {NEW_ASSIGNED_FILE_PATH}')

print('History file:', HISTORY_FILE_PATH)
print('New assigned file:', NEW_ASSIGNED_FILE_PATH)
print('Output folder:', MERGE_OUTPUT_DIR)

History file: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\lead_90th_history_no_duplicate_renamed.csv
New assigned file: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\assign_pipeline\Public_Water_Supply_90th_Percentiles_2025_with_assigned_pwsid.csv
Output folder: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\merge_pipeline


In [6]:
STANDARD_COLUMNS = [
    'pwsid',
    'system_name',
    'county',
    'population',
    'monitoring_end_date',
    'lead_90th_ppb',
    'includes_5th_liter_or_not',
    'sampling_next_due_subject_to_change',
    'year',
    'month',
    'above_action_level',
]


def load_clean_csv(path):
    df = pd.read_csv(path)
    missing = [col for col in STANDARD_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'Missing expected columns in {path.name}: {missing}')

    df = df[STANDARD_COLUMNS].copy()
    df['monitoring_end_date'] = pd.to_datetime(df['monitoring_end_date'], errors='coerce')
    df['sampling_next_due_subject_to_change'] = pd.to_datetime(
        df['sampling_next_due_subject_to_change'],
        errors='coerce',
    )

    for col in ['pwsid', 'system_name', 'county', 'includes_5th_liter_or_not']:
        df[col] = df[col].astype(str).str.strip()

    return df


def groupby_pwid_review(df):
    duplicate_checking = df.copy()
    duplicate_checking['lead_value_count'] = (
        duplicate_checking
        .groupby(['pwsid', 'monitoring_end_date'])['lead_90th_ppb']
        .transform('nunique')
    )
    duplicate_checking['system_name_count'] = (
        duplicate_checking
        .groupby(['pwsid', 'monitoring_end_date'])['system_name']
        .transform('nunique')
    )

    need_review = duplicate_checking[
        duplicate_checking['lead_value_count'] > 1
    ].sort_values(['pwsid', 'monitoring_end_date', 'system_name']).reset_index(drop=True)

    return duplicate_checking, need_review


In [7]:
history_df = load_clean_csv(HISTORY_FILE_PATH)
new_assigned_df = load_clean_csv(NEW_ASSIGNED_FILE_PATH)

merged_raw = pd.concat([history_df, new_assigned_df], ignore_index=True)
exact_duplicate_count_before = int(merged_raw.duplicated().sum())

duplicate_rows = merged_raw[
    merged_raw.duplicated(keep=False)
].sort_values(['pwsid', 'monitoring_end_date', 'system_name']).reset_index(drop=True)

merged_clean = merged_raw.drop_duplicates().sort_values(
    ['pwsid', 'monitoring_end_date', 'system_name']
).reset_index(drop=True)

exact_duplicate_count_after = int(merged_clean.duplicated().sum())
merged_duplicate_checking, merged_need_review = groupby_pwid_review(merged_clean)

summary_df = pd.DataFrame([
    {
        'history_rows': len(history_df),
        'new_assigned_rows': len(new_assigned_df),
        'merged_rows_before_drop_duplicates': len(merged_raw),
        'exact_duplicate_count_before': exact_duplicate_count_before,
        'merged_rows_after_drop_duplicates': len(merged_clean),
        'exact_duplicate_count_after': exact_duplicate_count_after,
        'need_review_rows_after_merge': len(merged_need_review),
    }
])

display(summary_df)
display(duplicate_rows.head())
display(merged_need_review.head())


,history_rows,new_assigned_rows,merged_rows_before_drop_duplicates,exact_duplicate_count_before,merged_rows_after_drop_duplicates,exact_duplicate_count_after,need_review_rows_after_merge
0,6043,1385,7428,699,6729,0,0


,pwsid,system_name,county,population,monitoring_end_date,lead_90th_ppb,includes_5th_liter_or_not,sampling_next_due_subject_to_change,year,month,above_action_level
0,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,GRAND TRAVERSE,128.0,2023-12-31,0.0,N,2026-09-30,2023.0,12.0,False
1,MI0000011,ACME TOWNSHIP - HOPE VILLAGE,GRAND TRAVERSE,128.0,2023-12-31,0.0,N,2026-09-30,2023.0,12.0,False
2,MI0000030,ADDISON,LENAWEE,605.0,2023-12-31,2.0,N,2026-09-30,2023.0,12.0,False
3,MI0000030,ADDISON,LENAWEE,605.0,2023-12-31,2.0,N,2026-09-30,2023.0,12.0,False
4,MI0000040,ADRIAN,LENAWEE,23663.0,2023-12-31,0.0,N,2026-09-30,2023.0,12.0,False


,pwsid,system_name,county,population,monitoring_end_date,lead_90th_ppb,includes_5th_liter_or_not,sampling_next_due_subject_to_change,year,month,above_action_level,lead_value_count,system_name_count


In [8]:
duplicate_rows_path = MERGE_OUTPUT_DIR / 'duplicate_rows_before_drop.csv'
need_review_path = MERGE_OUTPUT_DIR / 'merged_need_review.csv'
final_output_path = MERGE_OUTPUT_DIR / 'All_history_data_clean.csv'

duplicate_rows.to_csv(duplicate_rows_path, index=False, date_format='%Y-%m-%d')
merged_need_review.to_csv(need_review_path, index=False, date_format='%Y-%m-%d')

print('Saved:', duplicate_rows_path)
print('Saved:', need_review_path)

if exact_duplicate_count_after == 0 and len(merged_need_review) == 0:
    merged_clean.to_csv(final_output_path, index=False, date_format='%Y-%m-%d')
    print('Saved:', final_output_path)
else:
    print('All_history_data_clean.csv was not saved because duplicate or review issues still exist.')


Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\merge_pipeline\duplicate_rows_before_drop.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\merge_pipeline\merged_need_review.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\Pipeline\merge_pipeline\All_history_data_clean.csv
